# Self-Hosted LLMs: CPU 3B vs GPU 70B

Two models running on **your own EKS cluster** — no external API, no keys:

| Backend | Model | Where | Endpoint |
|---------|-------|-------|----------|
| CPU | `llama3.2` (3B) | c-family node | `ollama.ollama:11434` |
| GPU | `llama3.3:70b` (42 GB) | L40S / g6e.2xlarge | `ollama-gpu.ollama:11434` |

This notebook walks through three things:
1. Asking the 70B *where it runs* (spoiler: it has no idea)
2. A quality showcase (explain a hard idea simply)
3. **3B vs 70B side-by-side** — feel the quality jump

**Prerequisite:** run this inside JupyterHub, with both the CPU Ollama and the GPU `ollama-gpu` deployed. The endpoints are in-cluster DNS names, so they only resolve from a pod in the cluster.

In [ ]:
import requests, time

CPU_HOST, CPU_MODEL = "ollama.ollama:11434", "llama3.2"        # 3B on CPU
GPU_HOST, GPU_MODEL = "ollama-gpu.ollama:11434", "llama3.3:70b"  # 70B on the L40S

def ask(host, model, prompt, timeout=300):
    """Send a prompt to an in-cluster Ollama backend; return (response, seconds)."""
    t0 = time.time()
    r = requests.post(
        f"http://{host}/api/generate",
        json={"model": model, "prompt": prompt, "stream": False},
        timeout=timeout,
    )
    r.raise_for_status()
    return r.json()["response"].strip(), time.time() - t0

print("Ready. CPU =", CPU_MODEL, "| GPU =", GPU_MODEL)

## 1. Does the model know where it runs?

A useful reality check about what an LLM actually knows.

In [ ]:
resp, dt = ask(GPU_HOST, GPU_MODEL, "Tell me where this model is running.")
print(resp)
print(f"\n[70B on GPU · {dt:.1f}s]")

**The lesson:** it'll say something like *"a remote server somewhere... not publicly disclosed."* It's wrong — it's on **your L40S, in your EKS cluster, in us-east-1a**. An LLM only knows its *training data* + the *prompt*. Its own deployment isn't in either, so it pattern-matches to a generic "I'm a cloud model" answer. The real answer lives in your infrastructure (`kubectl`), not the model's weights.

## 2. Quality showcase — explain a hard idea simply

In [ ]:
resp, dt = ask(GPU_HOST, GPU_MODEL, "Explain quantum entanglement to a 10-year-old.")
print(resp)
print(f"\n[70B on GPU · {dt:.1f}s]")

## 3. 3B vs 70B, side by side

The same prompt to both models. This trick question (answer: **9**) is a nice discriminator — the small model often misreads it as subtraction (17 − 9 = 8); the 70B usually reasons it correctly.

Watch two things: the **answer quality** *and* the **latency** (3B-on-CPU vs 70B-on-GPU — the GPU often wins per-token despite being 23× bigger).

In [ ]:
prompt = (
    "A farmer has 17 sheep. All but 9 run away. "
    "How many are left? Explain your reasoning."
)

cpu_resp, cpu_dt = ask(CPU_HOST, CPU_MODEL, prompt)
gpu_resp, gpu_dt = ask(GPU_HOST, GPU_MODEL, prompt)

print("=" * 70)
print(f"3B on CPU  ({cpu_dt:.1f}s)\n" + "-" * 70)
print(cpu_resp)
print("=" * 70)
print(f"70B on GPU ({gpu_dt:.1f}s)\n" + "-" * 70)
print(gpu_resp)
print("=" * 70)
print("Correct answer: 9  ('all but 9 run away' = 9 stay)")

## Try your own

Swap in any prompt and compare. Reasoning, coding, and multi-step questions show the biggest gap; simple lookups look similar.

In [ ]:
my_prompt = "Write a haiku about running a 70B model on your own GPU."

for label, host, model in [("3B/CPU", CPU_HOST, CPU_MODEL), ("70B/GPU", GPU_HOST, GPU_MODEL)]:
    resp, dt = ask(host, model, my_prompt)
    print(f"--- {label} ({dt:.1f}s) ---\n{resp}\n")

---

You just compared a 3B and a 70B model **both self-hosted on your own cluster** — the small one on a CPU node, the large one on an L40S GPU. That's the core skill: knowing which model your hardware can hold, and feeling why the bigger one is worth the GPU.

*Part of the [AI-ML Unified Playground](https://github.com/suvmaha/ai-ml-unified-playground-platform) — see the Ollama hub at `/ollama.html` for local vs cloud modes.*